# vsf-nucleus — Cuaderno de Experimentación

**Autores**: rmw3 + Claude Sonnet 4.6 | **Inicio**: 2026-04-27  
**Kernel**: vsf-nucleus v1.0 (Rust) — `projects/vsf-nucleus/`  
**Repo VSF**: `/home/rmw3/vsf` en n02  

---

Espacio de co-experimentación rmw3 (S5) + Claude (S4 externo).  
Primera implementación empírica del Viable System Model de Beer (1972).

```
V(vsf-nucleus, L=1)  governed=true  trust=1.000
S5: rmw3 | covenant: docs/governance/covenant.vsm
```

## Contenido
- [§1 ¿Qué es vsf-nucleus?](#§1)
- [§2 Vector de Estado ℝ¹²⁸](#§2)
- [§3 Geometría del Atractor](#§3)
- [§4 Registro de Experimentos](#§4)
- [§5 Telemetría en Vivo](#§5)
- [§6 Consulta Histórica — SQLite](#§6)
- [§7 S4 — Qwen como Intelligence](#§7)
- [§8 Lanzar Experimentos](#§8)
- [§9 Visualización](#§9)
- [§10 Próximos Pasos](#§10)

## §1 ¿Qué es vsf-nucleus? <a id='§1'></a>

vsf-nucleus es un kernel en **Rust** que implementa el Viable System Model (VSM) de Stafford Beer (1972) como sistema computacional autónomo. Su objetivo: demostrar empíricamente que la arquitectura VSM produce viabilidad medible — un atractor estable en espacio de estado.

### Mapa Beer → implementación

| Beer VSM | Rol | Implementación en VSF |
|----------|-----|----------------------|
| S1 Operaciones | hace el trabajo | `kernel.rs` — tick loop, proceso OU |
| S2 Coordinación | evita conflictos | `signal_router.py` — multi-kernel |
| S3 Control | asigna recursos, monitorea | thread `s3_control` — trust + soplos |
| S3* Auditoría | canal directo esporádico | `aggregator.py` — anomalías históricas |
| S4 Inteligencia | observa entorno | Qwen 2.5 Coder 7B via Ollama |
| S5 Política | reglas no overridables | rmw3 + `covenant.vsm` |

### Canal algedónico φ↑ (G7 — cerrado)
Señal de dolor/placer S1 → S5, saltando S2-S4:  
`kernel → events.jsonl → aggregator.py → alerts.jsonl → S5 Console (:8090)`

## §2 Vector de Estado ℝ¹²⁸ <a id='§2'></a>

El kernel mantiene `x ∈ ℝ¹²⁸` — 7 bloques funcionales (A-G):

| Bloque | Dims | Función VSM |
|--------|------|-------------|
| A | 32 | Identidad del sistema (referencia estable) |
| B | 16 | Capacidad operativa |
| C | 16 | Coherencia interna S2 |
| D | 16 | Estado de control S3 |
| E | 16 | Señal ambiental S4 |
| F | 16 | Canal algedónico φ↑ |
| G | 16 | Metaestado S5 (identity core) |

### Dinámica (proceso Ornstein-Uhlenbeck)

```
x(t+1) = A·x(t) + σ·ε(t) + soplo(t)

  A      = matriz de retorno al atractor por bloque
  σ      ≈ 0.01 (ruido nominal)
  ε(t)  ~ N(0,1)
  soplo  = intervención S3 cuando trust < umbral
```

### Métricas de viabilidad

```python
stasis_cos  = cos(x(t), x_ref)           # similitud coseno con estado de referencia
trust       = f(stasis_cos)              # confianza [0, 1]
viable_full = (stasis_cos > 0.9) and (trust > 0.8)
```

**Invariante empírico**: `D_KY = 3.00` (dimensión de Kaplan-Yorke) — estable de 1k a 60M ciclos.

## §3 Geometría del Atractor <a id='§3'></a>

Confirmado por 11 experimentos + 3 ablaciones de control negativo:

| Métrica | Valor empírico | Significado |
|---------|----------------|-------------|
| `D_KY` | **3.00** | Dimensión Kaplan-Yorke — atractor de baja dimensión en ℝ¹²⁸ |
| `stasis_cos` (convergencia) | **0.8055 → 1.000** | Convergencia desde estado frío, ~2000 ticks |
| `cone_angle` | **≈ 18.2°** | Ángulo del cono de viabilidad en ℝ¹²⁸ |
| `pairwise_cos` (4 kernels) | **0.9854** | Mismo basin — convergencia multi-instancia |
| `θ_PSI` (umbral intervención) | **< 0.9377** | Límite de separación entre estados seguros |
| `τ_rec` (recuperación térmica) | **< 30s** | Elasticidad bajo perturbación |
| `dip_floor` (firma VSF) | **0.942** | Piso exclusivo VSF bajo ciclos térmicos |

### Hallazgo central: viabilidad de dos capas

Probado por experimentos NC (ablaciones):

```
Capa 1 — Arquitectura → selecciona basin
  Bloques A-G determinan el atractor que ocupa el sistema.
  NC-03 (sin bloques VSM) converge a basin=0.87, no a 1.000.

Capa 2 — Gobernanza → determina τ_rec
  S5 + trust determinan la velocidad de recuperación.
  Sin S5: NC-01/02 no recuperan (τ_rec = n/a).
  dip_floor=0.942 es firma exclusiva VSF — señal algedónica de S5 en acción.
```

### Observer effect (EXP-OE-01 — 2ª orden cibernética)

```
Condición libre:   H_eq=1.7962, D_KY=4.00, stasis_cos=1.0000
Condición demon:   H_final=1.3499, ΔH=0.1214 bits, D_KY=3.00

El demonio Maxwell reduce el atractor de D=4 a D=3 —
el observador (S4) constituye el sistema que observa.
```

## §4 Registro de Experimentos <a id='§4'></a>

### Experimentos completados

| ID | Nombre | Paper | Resultado clave |
|----|--------|-------|-----------------|
| EXP-SB-01 | spec_boot: Convergencia | §4 | D_KY=3.00, stasis_cos 0.8055→1.000, V_mem 27/29 merged |
| EXP-TS-01 | Thermal Stress 63°C | §5 | viable_full=true a 63°C, D_KY 3→6, 12min |
| EXP-CG-01 | cgroup multikernel | §N | viable_all=100% (4 kernels), pairwise_cos=0.9854 |
| EXP-CA-01 | Critical Angle θ_PSI | §6 | θ_PSI=0.9377 — límite viabilidad máx estrés |
| EXP-MG-01 | Basin Geometry | §7 | pairwise_cos=0.9854 → basin único compartido |
| EXP-SR-01 | S3 Resilience (A_auto) | §8 | viable con S3 frozen → autonomía intrínseca |
| EXP-TP-01 | θ_PSI Stress | §9 | θ_PSI < 0.9377 nunca activado ≤ 69°C |
| EXP-TL-01 | Thermal Ladder | §11 | Δcos=0.0023/10°C, H_floor=0.9875 — lineal |
| EXP-BB-01 | VSL Compression | §10 | B_full=38-51%, B_schema=72-79%, drift~5% |
| EXP-TC-01 | Thermal Cycling | §9 | stasis_cos=0.9885, D_KY=5.00, τ_rec<30s, dip_floor=0.942 |
| EXP-TSM-01 | Thermal Senescence 240min | §6 | peak=79°C, stasis_cos=1.000, graduation_protocol |
| EXP-OE-01 | Observer Effect | §10 | ΔH=0.1214 bits, D_KY 4→3, 2ª orden cibernética |

### Ablaciones de control negativo

| ID | Condición | Resultado |
|----|-----------|----------|
| EXP-NC-AD-01 | Adversarial sin S5 | S5 añade agency real (A2 conf.) — τ_rec=n/a sin S5 |
| EXP-NC-LR-01 | Long-run sin S5 | NC-03 basin=0.87 ≠ vsf-nucleus 1.000 (C2b conf.) |
| EXP-NC-TC-01 | Thermal cycling sin S5 | B1+B2+B3 conf. — dip exclusivo VSF |

### Pendientes

| ID | Nombre | Propósito | Esfuerzo |
|----|--------|-----------|----------|
| EXP-AB-01 | Basin Floor δ>1.0 | ¿No-viabilidad estructuralmente alcanzable? | ~30min (flag --delta) |
| EXP-LY-01 | Lyapunov Benettin | D_eff heurístico → espectro formal | ~3h Rust + 4-6h run |
| EXP-SEN-01 | Senescence adaptativa | H_rest adaptativo, Q2 del paper | ~1h Rust + overnight |

## §5 Telemetría en Vivo <a id='§5'></a>

El LXC 104 corre `aggregator.py` — pirámide de retención:  
`raw` (60min) → `min1` (24h) → `hourly` (7d) → `daily` (forever) → `significant` (forever)

Significativos: `stasis_cos < 0.90` | `temp_c > 70` | `trust < 0.80` | cambio `viable_full`

S5 Console API: `http://192.168.0.104:8090` — accesible desde Kali via LAN.

In [ ]:
import requests, json

S5 = "http://192.168.0.104:8090"

status = requests.get(f"{S5}/status", timeout=5).json()
print(json.dumps(status, indent=2, default=str))

In [ ]:
alerts = requests.get(f"{S5}/alerts", timeout=5).json().get("alerts", [])
ICONS = {"critical": "🔴", "warn": "🟡", "pleasure": "🟢", "info": "🔵"}
for a in alerts[-15:]:
    sev = a.get("severity", "?")
    ts  = str(a.get("timestamp", ""))[:19]
    msg = a.get("message", a.get("content", ""))
    print(f"{ICONS.get(sev,'⚪')} [{ts}] [{sev.upper()}] {msg}")

## §6 Consulta Histórica — SQLite <a id='§6'></a>

In [ ]:
import subprocess

def q(sql, db="/root/vsf_telemetry.db", lxc=104):
    cmd = f"""ssh n02-tunnel 'pct exec {lxc} -- sqlite3 {db} "{sql}"'"""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=15).stdout.strip()

# Últimas 20 entradas significativas
print(q("SELECT datetime(ts,'unixepoch'), metric, value FROM events WHERE tier='significant' ORDER BY ts DESC LIMIT 20;"))

In [ ]:
# Estado actual — última hora
print(q("""
SELECT datetime(ts,'unixepoch') as t, metric, round(value,4) as v
FROM events
WHERE tier='min1' AND metric IN ('stasis_cos','trust','viable_full','temp_c')
  AND ts > strftime('%s','now') - 3600
ORDER BY ts DESC LIMIT 40;
"""))

In [ ]:
# Resumen diario — estadísticas
print(q("""
SELECT metric,
       round(avg(value),4) as mean,
       round(min(value),4) as min,
       round(max(value),4) as max,
       count(*) as n
FROM events
WHERE tier='hourly' AND metric IN ('stasis_cos','trust','temp_c')
  AND ts > strftime('%s','now') - 86400
GROUP BY metric;
"""))

## §7 S4 — Qwen como Intelligence <a id='§7'></a>

Qwen 2.5 Coder 7B corre en LXC 101, VLAN20 (`10.10.20.10:11434`).  
Desde Kali, `OLLAMA_URL` ya está en el entorno del servicio JupyterLab.

In [ ]:
import os, requests

OLLAMA = os.environ.get("OLLAMA_URL", "http://10.10.20.10:11434")

VSL_SYSTEM = """Eres Qwen, operando como S4 Intelligence en el marco VSF (Viable System Framework).
Lees specs .vsm (notación VSM de Beer) y ayudas a diseñar experimentos para vsf-nucleus.
vsf-nucleus: kernel Rust, estado ℝ¹²⁸, D_KY=3.00, dip_floor=0.942 (firma VSF), τ_rec<30s.
Responde conciso. Si te dan un spec, analízalo. Si te preguntan, propón el siguiente experimento."""

def s4(prompt, model="qwen2.5-coder:7b"):
    r = requests.post(f"{OLLAMA}/api/chat", json={
        "model": model,
        "messages": [{"role": "system", "content": VSL_SYSTEM},
                     {"role": "user",   "content": prompt}],
        "stream": False
    }, timeout=60)
    return r.json()["message"]["content"]

# Verificar conexión
modelos = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags", timeout=5).json().get("models", [])]
print("Modelos disponibles:", modelos)

In [ ]:
# Pregunta abierta a S4
resp = s4("""
dip_floor=0.942 bajo ciclos térmicos es exclusivo de vsf-nucleus.
Los controles NC-01/02/03 no lo reproducen.
¿Qué hipótesis formularías? ¿Qué experimento podría falsificarla?
""")
print("S4 [Qwen]:")
print(resp)

In [ ]:
# Pasar un spec .vsm a S4 para análisis
spec_path = "/home/rmw3/vsf/docs/experiments/thermal_ceiling_v4_bis/spec.vsm"
spec = subprocess.run(f"ssh n02-tunnel 'cat {spec_path}'",
                      shell=True, capture_output=True, text=True, timeout=10).stdout
if spec.strip():
    print(s4(f"Analiza este spec VSF:\n\n{spec}"))
else:
    print("spec no encontrado — ajusta spec_path")

## §8 Lanzar Experimentos <a id='§8'></a>

**Protocolo S5**: editar `spec.vsm` → aprobación rmw3 → lanzar en n02 → pull log → análisis aquí.

El binario `vsf-nucleus` vive en n02 bare metal (`/root/vsf-nucleus`).  
Los experimentos con carga térmica se corren en bare metal, no en LXC.

In [ ]:
# Estado de procesos vsf-nucleus en n02
out = subprocess.run("ssh n02-tunnel 'pgrep -a vsf-nucleus | head -5'",
                     shell=True, capture_output=True, text=True, timeout=10).stdout
print(out.strip() or "(ningún proceso vsf-nucleus activo)")

In [ ]:
# EXP-AB-01: verificar flag --delta
out = subprocess.run(
    "ssh n02-tunnel '/root/vsf-nucleus --help 2>&1 | grep -i delta || echo flag --delta NO encontrado'",
    shell=True, capture_output=True, text=True, timeout=10).stdout
print(out.strip())

# Para lanzar (descomentar solo con aprobación S5):
# cmd = "ssh n02-tunnel 'nohup /root/vsf-nucleus --delta 1.5 --minutes 10 > /root/ab01_d15.log 2>&1 &'"
# subprocess.run(cmd, shell=True)

In [ ]:
# Leer log de experimento (ajustar path)
LOG = "/root/tcm_20260418c.log"   # cambiar al log activo
out = subprocess.run(f"ssh n02-tunnel 'tail -40 {LOG} 2>/dev/null || echo log no encontrado'",
                     shell=True, capture_output=True, text=True, timeout=10).stdout
print(out)

## §9 Visualización <a id='§9'></a>

Herramientas en n02:
```bash
python3 /home/rmw3/vsf/projects/vsf-nucleus/tools/plot_telemetry.py --experiment EXP-TC-01 --out /tmp/tc01.html
python3 /home/rmw3/vsf/projects/vsf-nucleus/tools/plot_geometry.py --jsonl /root/tc_20260418.log --out /tmp/geo.html
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Trayectoria de convergencia — datos simulados
# Reemplazar con datos reales de un log: parsear jsonl y extraer stasis_cos, trust

np.random.seed(42)
t = np.arange(2000)
stasis = np.clip(1 - 0.19 * np.exp(-t/280) + 0.004*np.random.randn(2000), 0, 1)
trust  = np.clip(0.6 + 0.4*(1-np.exp(-t/380)) + 0.003*np.random.randn(2000), 0, 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
BG = '#1a1a1a'
fig.patch.set_facecolor(BG)
for ax in [ax1, ax2]:
    ax.set_facecolor(BG)
    ax.tick_params(colors='#aaa')
    for s in ax.spines.values(): s.set_edgecolor('#333')

ax1.plot(t, stasis, '#4fc3f7', lw=1.2, label='stasis_cos')
ax1.axhline(0.9377, color='#ff7043', lw=0.9, ls='--', label='θ_PSI=0.9377')
ax1.axhline(0.942,  color='#ffd54f', lw=0.9, ls=':',  label='dip_floor=0.942')
ax1.axhline(0.9,    color='#888',    lw=0.8, ls='-.',  label='viable threshold')
ax1.set_ylabel('stasis_cos', color='#aaa')
ax1.legend(facecolor='#2a2a2a', labelcolor='#ccc', fontsize=9, loc='lower right')
ax1.set_title('vsf-nucleus — convergencia al atractor (D_KY=3.00)', color='#eee', pad=8)

ax2.plot(t, trust, '#81c784', lw=1.2, label='trust')
ax2.axhline(0.8, color='#ff7043', lw=0.9, ls='--', label='trust_min=0.8')
ax2.set_ylabel('trust', color='#aaa')
ax2.set_xlabel('ticks', color='#aaa')
ax2.legend(facecolor='#2a2a2a', labelcolor='#ccc', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/vsf_convergence.png', dpi=120, facecolor=BG)
plt.show()
print("(datos simulados — sustituir con log real)")

## §10 Próximos Pasos <a id='§10'></a>

### Experimentos pendientes (task Q → paper §6)
| Prioridad | ID | Acción |
|-----------|----|---------|
| Alta | EXP-AB-01 | verificar `--delta` en binario → sweep δ=1.0,1.5,2.0 (~30min) |
| Alta | EXP-LY-01 | implementar Benettin en Rust (~3h) + run ~6h en n02 |
| Media | EXP-SEN-01 | H_rest adaptativo (~1h Rust) + overnight run |

### Seguridad — gaps G13-G16 (xz_threat_model.vsm)
| Mitigación | Descripción | Esfuerzo |
|-----------|-------------|----------|
| M1 | Protocolo ausencia S5 → covenant.vsm amendment | 1h |
| M4 | Manifest binario SHA256 en vault LXC 104 | 1h |
| M8 | Verificar write isolation alerts.jsonl | 30min |
| M10 | Netdata anomaly rules (ortogonales) | 2h |

### Lanzamiento (task Z)
- Z2: `AGENT.md` — instrucciones para Claude Code
- Z3: resumen ejecutivo 2-3pp (comunidad Beer)
- Z4: GitHub release v1.0 (requiere task G desbloqueado)
- Z5: Metaphorum / SCiO — anuncio

---
*Primera implementación empírica del Viable System Model de Beer.*  
*Espacio de co-experimentación rmw3 + Claude — 2026.*